## Setup

Before extracting the training data and building the model, we need to ensure the config is set up correctly.
- `default_aoi` or default area of interest should be populated as a thin coastal strip (about 14 km max inland).
- `target` should be set to GMW ([projects/sat-io/open-datasets/GMW/annual-extent/GMW_MNG_2020](https://gee-community-catalog.org/projects/mangrove/#earth-engine-snippet-sentinel-raster-and-vector-baseline-v4019))
    - `mangrove_class`: 1
    - `permanent_water_threshold`: 100
- Within `export` settings,
    - `num_sample_points`: 6000
    - `target_name`: 'label'
    - `scale`: 10
    - `seed`: 48
    - `tile_scale`: 4
    - `geometries`: True
- `default_date_range` should have a `start_date`: '2020-01-01' and `end_date`: '2020-12-31'

In [3]:
import sys

import ee

from mangrove_mapper import MangroveMapper

sys.path.insert(0, "../../../")

PROJECT = "irp-zu" # Insert your GEE project name
TRAINING_DATA = "training_data_coastwide"

ee.Authenticate()
ee.Initialize(project=PROJECT)

mm = MangroveMapper(project_name=PROJECT)
CONFIG = {key: mm.config[key] for key in ['default_aoi', 'target', 'export', 'default_date_range']}
CONFIG

{'default_aoi': [[-59.38308424806478, 8.054416523553055],
  [-58.884720106527595, 7.670222740307287],
  [-58.6066393915342, 7.3742767194813394],
  [-58.49646429770003, 6.993420918568921],
  [-57.771841640151585, 6.503265198524917],
  [-57.65508973640652, 6.412177452976993],
  [-57.568594569839085, 6.28012794549925],
  [-57.44877778487628, 6.245078372130237],
  [-57.23569020160864, 6.087540882000849],
  [-57.125826920358634, 5.866277255134532],
  [-57.11987172043847, 6.118209943117061],
  [-57.44671498215722, 6.410339167612445],
  [-58.10040150559472, 6.917750154191127],
  [-58.33111439621972, 7.168528757883949],
  [-58.63323841965722, 7.696886561702809],
  [-59.10290394700097, 8.132152633198771],
  [-59.69616566575097, 8.394445875718194],
  [-59.99279652512597, 8.549292721463692],
  [-59.53002638673666, 8.152306216451324]],
 'target': {'dataset_id': 'projects/sat-io/open-datasets/GMW/annual-extent/GMW_MNG_2020',
  'mangrove_class': 1,
  'permanent_water_threshold': 100},
 'export': {'n

In [6]:
from pathlib import Path

import pandas as pd
import rasterio

from mangrove_mapper.inference import extent_from_prob, extent_tile, mosaic, predict_tile, snap_to_domain
from mangrove_mapper.models import MangroveClassifier
from mangrove_mapper.sensors import ALOSPalsar, Landsat8, Sentinel1, Sentinel2
from mangrove_mapper.utils.fusion import concat_sensors, product_label
from mangrove_mapper.validation.validator import auc_scores, evaluate_model, olofsson_estimate, summarise_estimates

REBUILD = False    # True forces re-export / re-inference even if artifacts exist


YEAR = 2021 
START, END = f"{YEAR}-01-01", f"{YEAR}-12-31"
THRESHOLD = 0.50
TAG = int(THRESHOLD * 100)

CRS, SCALE, TILE = "EPSG:32621", 10, 2048 # TILE - the higher, the faster but more RAM usage 
DRIVE_FOLDER = "MangroveMapper_ML_Exports" # Name of Google Drive Folder

DATA = Path("../../../data")
COAST = DATA / "fullcoast"
COAST.mkdir(parents=True, exist_ok=True)
SHEET = Path("../../../mangrove_mapper/accuracy_assessment/"
             "full_coast_accuracy_assessment_final.csv")
GSW = COAST / "fullcoast-gsw-occurrence-30m.tif"
OUT_TABLE = Path("coastwide_results.csv")

PROB_MOSAIC = DATA / f"{YEAR}-COAST-PROB-XGB-GMW-full.tif"
EXTENT_MOSAIC = DATA / f"{YEAR}-COAST-EXTENT-XGB-GMW-full-T{TAG}.tif"
WC_RAW = DATA / f"{YEAR}-worldcover-v200-coast.tif"
WC_SNAP = DATA / f"{YEAR}-worldcover-v200-coast-snapped.tif"

mc = MangroveClassifier()
AOI = mm.aoi_polygon
print(f"REBUILD={REBUILD}")


REBUILD=False


## 1. Data Export & Model Training

There should be 35 feature bands and just under 12,000 rows (~6,000 per sample class)

In [8]:
# training samples (GMW labels, from config) — export if absent
TRAIN_CSV = DATA / f"{TRAINING_DATA}.csv"
if REBUILD or not TRAIN_CSV.exists():
    # presence_only=True because the config target (GMW) is presence-only;
    # AOI, date range and points-per-class all come from mm.config
    mm.get_sensor_data(TRAINING_DATA, retrieve_from_drive=True, presence_only=True)
train = pd.read_csv(TRAIN_CSV)
FEATS = [c for c in train.columns if c not in mc.NON_FEATURES]
print(f"{len(train)} training rows | {len(FEATS)} features")


11992 training rows | 35 features


The final model should be the XGBoost model trained on GMW training labels with 6000 samples per class

In [22]:
# model: load the trained XGB-GMW if present, else train on the samples + save
from mangrove_mapper.utils.regions import tag_regions

MODEL_PATH = Path("../../../trained_models") / "xgb_6k_gmw_full_fusion.json"
if REBUILD or not MODEL_PATH.exists():
    regions_data_path = (DATA / "guyana_regions.geojson") if (DATA / "guyana_regions.geojson").exists() else None
    tr = tag_regions(train, regions_data_path=regions_data_path)
    X, y, groups = mc.features_except(tr), tr["label"].values, tr["region"].values
    model, params = mc.build_final_model(X, y, groups=groups,
                                         base_estimator="xgboost", param_dist=mc.XGB_PARAMS_DIST)
    mc.save_model(model, name=MODEL_PATH.stem, best_params=params)
model = mc.load_model(str(MODEL_PATH))
print(f"model: {MODEL_PATH.name}")


Exported guyana_regions.geojson at this path: data/guyana_regions.geojson


/opt/miniconda3/envs/irp2026/lib/python3.14/site-packages/geopandas/array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Chosen hyperparameters: {'subsample': 1.0, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.5}
model: xgb_6k_gmw_full_fusion.json


In [9]:
# Earth Engine image builders
def coast_stack():
    sensors = [Landsat8(START, END, AOI, PROJECT), Sentinel2(START, END, AOI, PROJECT),
               Sentinel1(START, END, AOI, PROJECT), ALOSPalsar(START, END, AOI, PROJECT)]
    return concat_sensors(sensors).toFloat().select(FEATS).clip(AOI)


def gsw_image():
    import ee
    return (ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")
            .unmask(0).clip(AOI).toUint8())


## 2. Exporting Tiled Coast and Water Mask

For Guyana's entire coast (458 km) at 10 m resolution with an inland distance of around 10 km, the total pixel count of the area of interest polygon is on the order of magnitude $10^7$. This is why tiling was used (not as important on a regional level). Given this, exporting the data as a raster may take quite long in Google Earth Engine (GEE), around 1hr/2hrs in my experience, and will take up several GBs of memory. 

As a result, I have saved a copies of these tiles to my Imperial One Drive, which you can freely download and bypass these wait times if you'd like. 

fullcoast-2021-image10m => [Download here](https://imperiallondon-my.sharepoint.com/:f:/g/personal/zam25_ic_ac_uk/IgAVDM_ugdfJT7bPInaNKWoXAUxOxvr0oNdqxN-pNHVnkLw?e=QeNj9T)  

fullcoast-gsw-occurrence-30m => [Download here](https://imperiallondon-my.sharepoint.com/:f:/g/personal/zam25_ic_ac_uk/IgBFGKbTjMSZRIVCrr0AkeiaAXr6rF5BYmOMLyCU08KoQQ4?e=Z2cF64)  

fullcoast-gsw-occurence-30m.tif (the merged gsw tiles, equivalent to `GSW`) => [Download here](https://imperiallondon-my.sharepoint.com/:i:/g/personal/zam25_ic_ac_uk/IQDYa9Ck3XlURaA8yJrWzWlkAROJmBLwZAXBjqHu77jBTY0?e=SRkUSD)  


Please ensure the downloaded files go inside `PROJECT_ROOT/data/fullcoast` excluding the original folder they were contained in. 

If you are going ahead with a fresh export from GEE, you can monitor your tasks in the [Earth Engine Task Manager](https://code.earthengine.google.com/tasks). Once they are downloaded, they will appear in your Google Drive under the folder name associated with `DRIVE_FOLDER`. You will need to download the files to your local disk and port them inside `PROJECT_ROOT/data/fullcoast`. 

In [10]:
# feature-stack + GSW tiles (each built independently if absent)
stacks = sorted(COAST.glob(f"fullcoast-{YEAR}-image10m*.tif"))

if REBUILD or not PROB_MOSAIC.exists():  
    ct = mm.pinned_transform(crs=CRS, scale=SCALE) # used to ensure all rasters are aligned to same grid

    # feature stack: export only if its tiles are absent
    if REBUILD or not stacks:
        mm.export_image_tiled(coast_stack(), f"fullcoast-{YEAR}-image10m",
                              crs_transform=ct, crs=CRS, tile=TILE,
                              folder=DRIVE_FOLDER, output_dir=COAST)
        stacks = sorted(COAST.glob(f"fullcoast-{YEAR}-image10m*.tif"))

    # GSW: export only if neither the merged raster nor its tiles are present
    gsw_tiles = sorted(COAST.glob("fullcoast-gsw-occurrence-30m-*.tif"))
    if REBUILD or (not GSW.exists() and not gsw_tiles):
        mm.export_image_tiled(gsw_image(), "fullcoast-gsw-occurrence-30m",
                              scale=30, crs=CRS, tile=TILE,
                              folder=DRIVE_FOLDER, output_dir=COAST)

    # merge GSW tiles once
    if REBUILD or not GSW.exists():
        mosaic(sorted(COAST.glob("fullcoast-gsw-occurrence-30m-*.tif")), GSW, nodata=0)

print(f"{len(stacks)} stack tiles | prob mosaic: {PROB_MOSAIC.exists()}")


Tiled image export started: fullcoast-2021-image10m (task SPY764BM2TUO56YCFUGPPB6Y)
Waiting for export to complete...
See <a href='https://code.earthengine.google.com/tasks'>Earth Engine Task Manager</a> for more information about your task
  READY...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNING...
  RUNNI

## 3. Inference (Probability Mosaic Raster)

At this point, you should have 48 tiles covering the entire coast with feature bands associated with every pixel.
If you're running this for the first time, you likely do not have a probability raster mosaic (`prob mosaic`) yet. 

In [29]:
# windowed inference (build tiles if absent)
def tile_id(stack_path):
    return stack_path.stem.split(f"{YEAR}-image10m", 1)[-1].lstrip("-") or "single"


if REBUILD or not PROB_MOSAIC.exists():
    for stack in sorted(COAST.glob(f"fullcoast-{YEAR}-image10m*.tif")):
        tid = tile_id(stack)
        prob = COAST / f"{YEAR}-prob-xgb-gmw-{tid}.tif"
        extent = COAST / f"{YEAR}-EXTENT-XGB-GMW-full-T{TAG}-{tid}.tif"
        if REBUILD or not prob.exists():
            predict_tile(model, FEATS, stack, prob)
        if REBUILD or not extent.exists():
            extent_tile(prob, stack, GSW, THRESHOLD, extent)
    print("inference done")
else:
    print("prob mosaic exists...skipping inference")


inference done


In [31]:
# mosaic tiles => coast rasters (build if absent)
if REBUILD or not PROB_MOSAIC.exists():
    mosaic(sorted(COAST.glob(f"{YEAR}-prob-xgb-gmw-*.tif")), PROB_MOSAIC)
if REBUILD or not EXTENT_MOSAIC.exists():
    mosaic(sorted(COAST.glob(f"{YEAR}-EXTENT-XGB-GMW-full-T{TAG}-*.tif")), EXTENT_MOSAIC)

for p in (PROB_MOSAIC, EXTENT_MOSAIC):
    with rasterio.open(p) as s:
        print(f"{p.name}: {s.width:,} x {s.height:,} px")


2021-COAST-PROB-XGB-GMW-full.tif: 31,624 x 29,789 px
2021-COAST-EXTENT-XGB-GMW-full-T50.tif: 31,624 x 29,789 px


With a probability raster in place, extent maps can be generated at different threshold without model retraining and inference.

## 4. Accuracy Estimates 

We now have an extent map by which we can use the olofsson estimator to determine user's, producer's, overall accuracy and error-adjusted area estimates.

In [19]:
# model accuracy (design-based)
df_eval, meta = evaluate_model(str(SHEET), model_name="xgb_gmw_t50",
                              model_extent_map_path=str(EXTENT_MOSAIC))
print(f"domain {meta['domain_ha']:,.0f} ha | weights {meta['weights']}")

model_results = {ref: olofsson_estimate(df_eval, map_col="xgb_gmw_t50",
                                        weights=meta["weights"], domain_ha=meta["domain_ha"],
                                        ref_col=ref)
                 for ref in ("ceo_label", "planet_label")}
for ref, res in model_results.items():
    print(ref)
    display(summarise_estimates({"xgb_gmw_t50": res}, focus="mangrove"))


domain 668,684 ha | weights {0: 0.95581587447764, 1: 0.044184125522359924}
ceo_label


,n,overall acc.,mangrove UA,mangrove PA,mangrove area (ha)
xgb_gmw_t50,600,0.972 ± 0.007,0.440 ± 0.056,0.859 ± 0.238,15130.360 ± 4494.442


planet_label


,n,overall acc.,mangrove UA,mangrove PA,mangrove area (ha)
xgb_gmw_t50,600,0.963 ± 0.011,0.390 ± 0.055,0.643 ± 0.261,17914.024 ± 7391.065


In [20]:
# threshold sweep (build extents if absent)
THRESH = [0.50, 0.70, 0.80]
sweep = {ref: {} for ref in ("ceo_label", "planet_label")}
for t in THRESH:
    tag = f"t{int(t * 100)}"
    ext_t = DATA / f"{YEAR}-COAST-EXTENT-XGB-GMW-full-{tag}.tif"
    if REBUILD or not ext_t.exists():
        extent_from_prob(PROB_MOSAIC, EXTENT_MOSAIC, t, ext_t)
    d, m = evaluate_model(str(SHEET), model_name=tag, model_extent_map_path=str(ext_t))
    for ref in sweep:
        sweep[ref][tag] = olofsson_estimate(d, map_col=tag, weights=m["weights"],
                                            domain_ha=m["domain_ha"], ref_col=ref)
for ref, res in sweep.items():
    print(ref)
    display(summarise_estimates(res, focus="mangrove"))


ceo_label


,n,overall acc.,mangrove UA,mangrove PA,mangrove area (ha)
t50,600,0.972 ± 0.007,0.440 ± 0.056,0.859 ± 0.238,15130.360 ± 4494.442
t70,600,0.964 ± 0.014,0.496 ± 0.061,0.528 ± 0.200,23775.385 ± 9047.829
t80,600,0.959 ± 0.016,0.557 ± 0.066,0.429 ± 0.153,29815.448 ± 10541.846


planet_label


,n,overall acc.,mangrove UA,mangrove PA,mangrove area (ha)
t50,600,0.963 ± 0.011,0.390 ± 0.055,0.643 ± 0.261,17914.024 ± 7391.065
t70,600,0.962 ± 0.014,0.445 ± 0.061,0.501 ± 0.202,22490.365 ± 9046.243
t80,600,0.957 ± 0.016,0.498 ± 0.066,0.401 ± 0.151,28464.945 ± 10543.240


Non-mangrove estimates (not reported). Confusion matrix may appear in appendix

In [21]:
for ref, res in sweep.items():
    print(ref)
    display(summarise_estimates(res, focus="non_mangrove"))

ceo_label


,n,overall acc.,non_mangrove UA,non_mangrove PA,non_mangrove area (ha)
t50,600,0.972 ± 0.007,0.997 ± 0.007,0.975 ± 0.002,653553.690 ± 4494.442
t70,600,0.964 ± 0.014,0.983 ± 0.014,0.980 ± 0.002,644908.665 ± 9047.829
t80,600,0.959 ± 0.016,0.974 ± 0.016,0.984 ± 0.002,638868.602 ± 10541.846


planet_label


,n,overall acc.,non_mangrove UA,non_mangrove PA,non_mangrove area (ha)
t50,600,0.963 ± 0.011,0.990 ± 0.011,0.972 ± 0.002,650770.026 ± 7391.065
t70,600,0.962 ± 0.014,0.983 ± 0.014,0.978 ± 0.002,646193.685 ± 9046.243
t80,600,0.957 ± 0.016,0.974 ± 0.016,0.982 ± 0.002,640219.105 ± 10543.240


## 5. ROC/PR AUC Estimates

We also have threshold-independent estimates such as the Area Under the Curve (AUC) for Precision-Recall (PR) curve and Receiver Operating Characteristic (ROC) curve. 

In [24]:
# threshold-free AUC
auc = auc_scores(str(SHEET), str(PROB_MOSAIC), ref_cols=("ceo_label", "planet_label"))
display(pd.DataFrame(auc).T)


,roc_auc,pr_auc,n,base_rate
ceo_label,0.948391,0.836387,600.0,0.221667
planet_label,0.936259,0.774891,600.0,0.200000


## 6. WorldCover v200 Estimate

Exporting WorldCover v200 extent map, snapping it to the grid and estimating the Olofsson acurracies and areas

In [25]:
# WorldCover v200 baseline (build if absent)
if REBUILD or not WC_SNAP.exists():
    mm.export_image_tiled(product_label("ESA/WorldCover/v200", AOI, mangrove_class=95),
                          f"{YEAR}-worldcover-v200-coast",
                          crs_transform=mm.pinned_transform(crs=CRS, scale=SCALE),
                          crs=CRS, tile=TILE, folder=DRIVE_FOLDER, output_dir=COAST)
    mosaic(sorted(COAST.glob(f"{YEAR}-worldcover-v200-coast-*.tif")), WC_RAW)
    snap_to_domain(WC_RAW, EXTENT_MOSAIC, WC_SNAP)

dwc, mwc = evaluate_model(str(SHEET), model_name="worldcover_v200",
                          model_extent_map_path=str(WC_SNAP))
wc_results = {ref: olofsson_estimate(dwc, map_col="worldcover_v200", weights=mwc["weights"],
                                     domain_ha=mwc["domain_ha"], ref_col=ref)
              for ref in ("ceo_label", "planet_label")}
for ref, res in wc_results.items():
    print(ref)
    display(summarise_estimates({"worldcover_v200": res}, focus="mangrove"))


ceo_label


,n,overall acc.,mangrove UA,mangrove PA,mangrove area (ha)
worldcover_v200,600,0.959 ± 0.016,0.752 ± 0.068,0.415 ± 0.123,37517.019 ± 11015.875


planet_label


,n,overall acc.,mangrove UA,mangrove PA,mangrove area (ha)
worldcover_v200,600,0.966 ± 0.014,0.694 ± 0.072,0.472 ± 0.148,30478.292 ± 9518.654


## 7. Report Figure

latex table that appears in the final report

In [29]:
# assemble coast-wide table (point estimate ± 95% CI)
def pm(v, ci, fmt="{:.3f}"):
    return f"{fmt.format(v)} ± {fmt.format(ci)}"

rows = []
for name, res in {"XGB-GMW": model_results, "WorldCover v200": wc_results}.items():
    for ref, r in res.items():
        pc = r["per_class"].loc["mangrove"]
        row = {"map": name, "reference": ref.replace("_label", ""), "n": r["n"],
               "overall_acc": pm(r["overall_accuracy"], r["overall_accuracy_ci"]),
               "mangrove_UA": pm(pc["users_acc"], pc["users_ci"]),
               "mangrove_PA": pm(pc["producers_acc"], pc["producers_ci"]),
               "area_ha": pm(pc["area_ha"], pc["area_ha_ci"], "{:,.0f}")}
        if name.startswith("XGB"):   # AUC is model-only; auc_scores returns no CI
            row["roc_auc"] = f"{auc[ref]['roc_auc']:.3f}"
            row["pr_auc"] = f"{auc[ref]['pr_auc']:.3f}"
        rows.append(row)

table = pd.DataFrame(rows)
table


,map,reference,n,overall_acc,mangrove_UA,mangrove_PA,area_ha,roc_auc,pr_auc
0,XGB-GMW,ceo,600,0.972 ± 0.007,0.440 ± 0.056,0.859 ± 0.238,"15,130 ± 4,494",0.948,0.836
1,XGB-GMW,planet,600,0.963 ± 0.011,0.390 ± 0.055,0.643 ± 0.261,"17,914 ± 7,391",0.936,0.775
2,WorldCover v200,ceo,600,0.959 ± 0.016,0.752 ± 0.068,0.415 ± 0.123,"37,517 ± 11,016",NaN,NaN
3,WorldCover v200,planet,600,0.966 ± 0.014,0.694 ± 0.072,0.472 ± 0.148,"30,478 ± 9,519",NaN,NaN


In [34]:
# LaTeX for the report: drop constant n, rename headers, ± to $\pm$, prettify table
pretty = table.drop(columns="n").rename(columns={
    "map": "Map", "reference": "Reference",
    "overall_acc": "OA", "mangrove_UA": "UA", "mangrove_PA": "PA",
    "area_ha": "Area (ha)", "roc_auc": "ROC-AUC", "pr_auc": "PR-AUC"})
pretty = pretty.apply(lambda col: col.map(
    lambda x: x.replace("±", r"$\pm$") if isinstance(x, str) else x))

# highlight the best model (XGB-GMW)
bold_rows = pretty["Map"].str.startswith("XGB-GMW")
pretty.loc[bold_rows] = pretty.loc[bold_rows].apply(
    lambda col: col.map(lambda x: rf"\textbf{{{x}}}" if isinstance(x, str) and x else x))

latex = pretty.to_latex(index=False, escape=False, na_rep="",
                        column_format="llllllrr",
                        caption=r"XGBoost and WorldCover v200 extent-map model comparisons from 600 stratified independent reference plots collected for the entire Guyana coastline, interpreted against CEO and Planet imagery",
                        label="tab:coastwide-results")

# centre the float and shrink it to the text width so it fits the page
latex = (latex
         .replace(r"\begin{table}", "\\begin{table}[htbp]\n\\centering")
         .replace(r"\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\n\\begin{tabular}")
         .replace(r"\end{tabular}", "\\end{tabular}\n\\end{adjustbox}"))

Path("coastwide_results.tex").write_text(latex)
print(latex)

\begin{table}[htbp]
\centering
\caption{XGBoost and WorldCover v200 extent-map model comparisons from 600 stratified independent reference plots collected for the entire Guyana coastline, interpreted against CEO and Planet imagery}
\label{tab:coastwide-results}
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{llllllrr}
\toprule
Map & Reference & OA & UA & PA & Area (ha) & ROC-AUC & PR-AUC \\
\midrule
\textbf{XGB-GMW} & \textbf{ceo} & \textbf{0.972 $\pm$ 0.007} & \textbf{0.440 $\pm$ 0.056} & \textbf{0.859 $\pm$ 0.238} & \textbf{15,130 $\pm$ 4,494} & \textbf{0.948} & \textbf{0.836} \\
\textbf{XGB-GMW} & \textbf{planet} & \textbf{0.963 $\pm$ 0.011} & \textbf{0.390 $\pm$ 0.055} & \textbf{0.643 $\pm$ 0.261} & \textbf{17,914 $\pm$ 7,391} & \textbf{0.936} & \textbf{0.775} \\
WorldCover v200 & ceo & 0.959 $\pm$ 0.016 & 0.752 $\pm$ 0.068 & 0.415 $\pm$ 0.123 & 37,517 $\pm$ 11,016 &  &  \\
WorldCover v200 & planet & 0.966 $\pm$ 0.014 & 0.694 $\pm$ 0.072 & 0.472 $\pm$ 0.148 & 30,478 $\pm$ 9